# Dataset Creation Notebook

This notebook provides a template for creating and preprocessing datasets.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Configuration

In [3]:
CSV_PATH = Path("combined_unique.csv")
PROCESSED_DIR = Path("processed_datasets")
PROCESSED_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH)
df.head()

,line,isMetaphor
0,விழியில் விழுந்து இதயம் நுழைந்து,0.0
1,இரவும் பகலும் உரசிக் கொள்ளும்,0.0
2,அந்திப் பொழுதினில் வந்துவிடு,0.0
3,உன் வெள்ளிக் கொலுசொலியை,0.0
4,நீ மல்லிகைப் பூவை சூடிக் கொண்டால்,0.0


## 4. Preprocessing

In [4]:
df_processed = df[["line", "isMetaphor"]].dropna().copy()
df_processed["isMetaphor"] = df_processed["isMetaphor"].astype(int)

metaphor_df = df_processed[df_processed["isMetaphor"] == 1].copy()
non_metaphor_df = df_processed[df_processed["isMetaphor"] == 0].copy()

marker_pattern = r"போல்|போல|போன்ற"
has_marker = metaphor_df["line"].str.contains(marker_pattern, na=False)
with_marker = metaphor_df[has_marker].copy()
without_marker = metaphor_df[~has_marker].copy()

non_metaphor_df["marker_type"] = "non_metaphor"
with_marker["marker_type"] = "with_marker"
without_marker["marker_type"] = "without_marker"

df_processed = pd.concat(
    [non_metaphor_df, with_marker, without_marker],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

In [10]:
len(non_metaphor_df), len(with_marker), len(without_marker)

(6679, 4250, 1466)

## 5. Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
RANDOM_STATE = 42

def split_stratified(dataset, stratify_col):
    train_df, temp_df = train_test_split(
        dataset,
        test_size=(1 - TRAIN_RATIO),
        random_state=RANDOM_STATE,
        stratify=dataset[stratify_col]
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=RANDOM_STATE,
        stratify=temp_df[stratify_col]
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

def build_binary_dataset(positive_df, negative_df, positive_name, negative_multiplier=1):
    positive_df = positive_df.copy()
    negative_df = negative_df.copy()

    positive_df["label"] = 1
    positive_df["dataset_type"] = positive_name
    negative_df["label"] = 0
    negative_df["dataset_type"] = "non_metaphor"

    dataset = pd.concat([positive_df, negative_df], ignore_index=True)
    dataset = dataset[["line", "label", "isMetaphor", "marker_type", "dataset_type"]]

    train_df, val_df, test_df = split_stratified(dataset, "label")

    train_pos = train_df[train_df["label"] == 1]
    train_neg = train_df[train_df["label"] == 0]
    target_neg_count = min(len(train_neg), len(train_pos) * negative_multiplier)
    train_neg_sampled = train_neg.sample(n=target_neg_count, random_state=RANDOM_STATE)
    train_balanced = pd.concat([train_pos, train_neg_sampled], ignore_index=True)
    train_balanced = train_balanced.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    return train_balanced, val_df, test_df

def build_three_class_dataset(non_metaphor_df, with_marker, without_marker):
    three_class = pd.concat(
        [non_metaphor_df.copy(), with_marker.copy(), without_marker.copy()],
        ignore_index=True
    )
    label_map = {
        "non_metaphor": 0,
        "with_marker": 1,
        "without_marker": 2,
    }
    three_class["label"] = three_class["marker_type"].map(label_map)
    three_class["dataset_type"] = three_class["marker_type"]
    three_class = three_class[["line", "label", "isMetaphor", "marker_type", "dataset_type"]]
    return split_stratified(three_class, "label")

with_marker_train, with_marker_val, with_marker_test = build_binary_dataset(
    with_marker,
    non_metaphor_df,
    positive_name="with_marker",
    negative_multiplier=1
)

without_marker_train, without_marker_val, without_marker_test = build_binary_dataset(
    without_marker,
    non_metaphor_df,
    positive_name="without_marker",
    negative_multiplier=2
)

three_class_train, three_class_val, three_class_test = build_three_class_dataset(
    non_metaphor_df,
    with_marker,
    without_marker
)

for name, split_df in {
    "with_marker_train": with_marker_train,
    "with_marker_val": with_marker_val,
    "with_marker_test": with_marker_test,
    "without_marker_train": without_marker_train,
    "without_marker_val": without_marker_val,
    "without_marker_test": without_marker_test,
    "three_class_train": three_class_train,
    "three_class_val": three_class_val,
    "three_class_test": three_class_test,
}.items():
    print(name)
    print(split_df["label"].value_counts().sort_index())
    print()

## 6. Save Processed Dataset

In [ ]:
with_marker_train.to_csv(PROCESSED_DIR / "with_marker_train.csv", index=False)
with_marker_val.to_csv(PROCESSED_DIR / "with_marker_val.csv", index=False)
with_marker_test.to_csv(PROCESSED_DIR / "with_marker_test.csv", index=False)

without_marker_train.to_csv(PROCESSED_DIR / "without_marker_train.csv", index=False)
without_marker_val.to_csv(PROCESSED_DIR / "without_marker_val.csv", index=False)
without_marker_test.to_csv(PROCESSED_DIR / "without_marker_test.csv", index=False)

three_class_train.to_csv(PROCESSED_DIR / "three_class_train.csv", index=False)
three_class_val.to_csv(PROCESSED_DIR / "three_class_val.csv", index=False)
three_class_test.to_csv(PROCESSED_DIR / "three_class_test.csv", index=False)

print("Dataset saved successfully.")
for path in sorted(PROCESSED_DIR.glob("*.csv")):
    print(path)

## 7. Kaggle MuRIL Experiment Setup

Run these cells on Kaggle with GPU enabled and Internet turned on. The setup trains three MuRIL models and evaluates each model on the requested in-domain and cross-domain test sets.

In [ ]:
# Kaggle only: install packages that may be missing.
# If a package is already installed, pip will reuse it.
!pip install -q transformers accelerate lime shap

In [ ]:
import json
import inspect
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
from lime.lime_text import LimeTextExplainer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 8. Load Training, Validation, and Testing Splits

The binary datasets use label `0 = non_metaphor`, `1 = metaphor`. The three-class dataset uses `0 = non_metaphor`, `1 = with_marker`, `2 = without_marker`.

In [ ]:
import os

# Auto-detect Kaggle environment so you don't have to change the path manually.
# On Kaggle: upload your processed_datasets/ folder as a dataset named 'muril-metaphor'.
_on_kaggle = os.path.exists("/kaggle")
DATA_DIR = (
    Path("/kaggle/input/muril-metaphor/processed_datasets")
    if _on_kaggle
    else Path("processed_datasets")
)
OUTPUT_DIR = (
    Path("/kaggle/working/muril_outputs")
    if _on_kaggle
    else Path("muril_outputs")
)
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 128
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
USE_FP16 = torch.cuda.is_available()

DATASETS = {
    "with_marker_binary": {
        "train": "with_marker_train.csv",
        "val":   "with_marker_val.csv",
        "test":  "with_marker_test.csv",
        "num_labels": 2,
        "id2label": {0: "non_metaphor", 1: "with_marker_metaphor"},
    },
    "without_marker_binary": {
        "train": "without_marker_train.csv",
        "val":   "without_marker_val.csv",
        "test":  "without_marker_test.csv",
        "num_labels": 2,
        "id2label": {0: "non_metaphor", 1: "without_marker_metaphor"},
    },
    "three_class": {
        "train": "three_class_train.csv",
        "val":   "three_class_val.csv",
        "test":  "three_class_test.csv",
        "num_labels": 3,
        "id2label": {0: "non_metaphor", 1: "with_marker", 2: "without_marker"},
    },
}

def load_csv(name):
    return pd.read_csv(DATA_DIR / name)

splits = {}
for dataset_name, cfg in DATASETS.items():
    splits[dataset_name] = {
        "train": load_csv(cfg["train"]),
        "val":   load_csv(cfg["val"]),
        "test":  load_csv(cfg["test"]),
    }
    print("\n", dataset_name)
    for split_name, split_df in splits[dataset_name].items():
        print(split_name, len(split_df), split_df["label"].value_counts().sort_index().to_dict())


## 9. MuRIL Trainer Helpers

In [ ]:
class MetaphorDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df["line"].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
        )
        encoded["labels"] = self.labels[idx]
        return encoded

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(logits, labels, weight=weights)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }


def make_training_args(output_dir):
    kwargs = {
        "output_dir": str(output_dir),
        "save_strategy": "epoch",
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": BATCH_SIZE,
        "num_train_epochs": EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "logging_dir": str(output_dir / "logs"),
        "logging_steps": 50,
        "save_total_limit": 2,
        "seed": SEED,
        "report_to": "none",
        "fp16": USE_FP16,
    }
    strategy_arg = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
    kwargs[strategy_arg] = "epoch"
    return TrainingArguments(**kwargs)


def train_muril_model(dataset_name):
    cfg = DATASETS[dataset_name]
    id2label = cfg["id2label"]
    label2id = {name: idx for idx, name in id2label.items()}

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=cfg["num_labels"],
        id2label=id2label,
        label2id=label2id,
    )

    train_df = splits[dataset_name]["train"]
    val_df = splits[dataset_name]["val"]

    train_dataset = MetaphorDataset(train_df, tokenizer, MAX_LENGTH)
    val_dataset = MetaphorDataset(val_df, tokenizer, MAX_LENGTH)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    class_ids = np.array(sorted(id2label.keys()))
    weights = compute_class_weight(
        class_weight="balanced",
        classes=class_ids,
        y=train_df["label"].astype(int).to_numpy(),
    )
    class_weights = torch.tensor(weights, dtype=torch.float)
    print(dataset_name, "class weights:", dict(zip(class_ids.tolist(), weights.round(4).tolist())))

    output_dir = OUTPUT_DIR / dataset_name
    trainer = WeightedTrainer(
        model=model,
        args=make_training_args(output_dir),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        class_weights=class_weights,
    )
    trainer.train()
    trainer.save_model(output_dir / "best_model")
    tokenizer.save_pretrained(output_dir / "best_model")
    return trainer, tokenizer

## 10. Evaluation Helpers

For binary models, cross-testing on the three-class test set is mapped into binary form: both `with_marker` and `without_marker` become label `1 = metaphor`.

In [ ]:
def prepare_eval_df(eval_df, target_dataset_name):
    """Re-cast labels for cross-domain evaluation.

    For binary models the ground-truth label is derived from `isMetaphor`
    (the original binary flag), mapping any metaphor subtype to label 1.
    This is intentional: when testing a binary model on the three-class
    test set we want to know whether it detects *any* metaphor, regardless
    of marker type.

    Note on non-metaphor overlap: the non-metaphor pool is split
    independently per dataset, so ~330-350 non-metaphor rows are shared
    between the binary and three-class test sets. This is expected and
    documented here for reviewers.
    """
    eval_df = eval_df.copy()
    if DATASETS[target_dataset_name]["num_labels"] == 2:
        eval_df["label"] = (eval_df["isMetaphor"].astype(int) == 1).astype(int)
    return eval_df


def evaluate_model(trainer, tokenizer, target_dataset_name, eval_df, eval_name):
    cfg = DATASETS[target_dataset_name]
    prepared_df = prepare_eval_df(eval_df, target_dataset_name)
    eval_dataset = MetaphorDataset(prepared_df, tokenizer, MAX_LENGTH)
    prediction_output = trainer.predict(eval_dataset)
    preds = np.argmax(prediction_output.predictions, axis=-1)
    labels = prepared_df["label"].astype(int).to_numpy()

    id2label = cfg["id2label"]
    label_names = [id2label[i] for i in sorted(id2label)]

    metrics = compute_metrics((prediction_output.predictions, labels))
    report = classification_report(
        labels,
        preds,
        labels=sorted(id2label),
        target_names=label_names,
        zero_division=0,
        output_dict=True,
    )
    cm = confusion_matrix(labels, preds, labels=sorted(id2label))

    print("\n===", eval_name, "===")
    print(json.dumps(metrics, indent=2))
    print(classification_report(
        labels,
        preds,
        labels=sorted(id2label),
        target_names=label_names,
        zero_division=0,
    ))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=label_names).plot(ax=ax, cmap="Blues", values_format="d")
    plt.xticks(rotation=30, ha="right")
    plt.title(eval_name)
    plt.tight_layout()

    result_dir = OUTPUT_DIR / target_dataset_name / "cross_eval"
    result_dir.mkdir(parents=True, exist_ok=True)
    safe_name = eval_name.replace(" ", "_").replace("/", "_").lower()

    # Save confusion matrix figure to disk (fix: was only shown inline before)
    fig.savefig(result_dir / f"{safe_name}_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    # Save metrics JSON
    with (result_dir / f"{safe_name}_metrics.json").open("w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    with (result_dir / f"{safe_name}_classification_report.json").open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    # Also write top-level test_metrics.json and classification_report.json
    # for the in-domain test result (first call per experiment), matching
    # the expected output locations from the project spec.
    top_dir = OUTPUT_DIR / target_dataset_name
    if not (top_dir / "test_metrics.json").exists():
        with (top_dir / "test_metrics.json").open("w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)
        with (top_dir / "classification_report.json").open("w", encoding="utf-8") as f:
            json.dump(report, f, indent=2)

    return {"metrics": metrics, "report": report, "confusion_matrix": cm, "predictions": preds}


## 11. Experiment 1

Train on `with_marker + non_metaphor`, then test on:

1. Similar `with_marker + non_metaphor` test data
2. `without_marker + non_metaphor` test data
3. `with_marker + without_marker + non_metaphor` test data

In [ ]:
with_marker_trainer, with_marker_tokenizer = train_muril_model("with_marker_binary")

exp1_results = {}
exp1_results["similar_with_marker_test"] = evaluate_model(
    with_marker_trainer,
    with_marker_tokenizer,
    "with_marker_binary",
    splits["with_marker_binary"]["test"],
    "Exp1 train with_marker: test with_marker binary",
)
exp1_results["without_marker_test"] = evaluate_model(
    with_marker_trainer,
    with_marker_tokenizer,
    "with_marker_binary",
    splits["without_marker_binary"]["test"],
    "Exp1 train with_marker: test without_marker binary",
)
exp1_results["three_class_test_binary_mapped"] = evaluate_model(
    with_marker_trainer,
    with_marker_tokenizer,
    "with_marker_binary",
    splits["three_class"]["test"],
    "Exp1 train with_marker: test three_class mapped to binary",
)

## 12. Experiment 2

Train on `without_marker + non_metaphor`, then test on:

1. Similar `without_marker + non_metaphor` test data
2. `with_marker + non_metaphor` test data
3. `with_marker + without_marker + non_metaphor` test data

In [ ]:
without_marker_trainer, without_marker_tokenizer = train_muril_model("without_marker_binary")

exp2_results = {}
exp2_results["similar_without_marker_test"] = evaluate_model(
    without_marker_trainer,
    without_marker_tokenizer,
    "without_marker_binary",
    splits["without_marker_binary"]["test"],
    "Exp2 train without_marker: test without_marker binary",
)
exp2_results["with_marker_test"] = evaluate_model(
    without_marker_trainer,
    without_marker_tokenizer,
    "without_marker_binary",
    splits["with_marker_binary"]["test"],
    "Exp2 train without_marker: test with_marker binary",
)
exp2_results["three_class_test_binary_mapped"] = evaluate_model(
    without_marker_trainer,
    without_marker_tokenizer,
    "without_marker_binary",
    splits["three_class"]["test"],
    "Exp2 train without_marker: test three_class mapped to binary",
)

## 13. Experiment 3

Train on `with_marker + without_marker + non_metaphor`, then test on:

1. Similar three-class test data
2. `without_marker + non_metaphor` test data
3. `with_marker + non_metaphor` test data

For binary test sets, the labels are remapped into the three-class label space before evaluation.

In [ ]:
def binary_to_three_class(eval_df, metaphor_class_label):
    """Remap a binary test set into the three-class label space.

    `metaphor_class_label` must be 1 (with_marker) or 2 (without_marker).
    The function asserts that all metaphor rows carry a single marker_type
    so a fixed label is safe; if the input ever changes this will surface
    immediately rather than silently producing wrong labels.
    """
    eval_df = eval_df.copy()
    metaphor_rows = eval_df[eval_df["isMetaphor"].astype(int) == 1]
    if len(metaphor_rows) > 0:
        unique_types = metaphor_rows["marker_type"].unique()
        assert len(unique_types) == 1, (
            f"binary_to_three_class expects a single metaphor subtype but found: {unique_types}. "
            "Pass a pure with_marker or without_marker test split."
        )
    eval_df["label"] = np.where(eval_df["isMetaphor"].astype(int) == 0, 0, metaphor_class_label)
    return eval_df

three_class_trainer, three_class_tokenizer = train_muril_model("three_class")

exp3_results = {}
exp3_results["similar_three_class_test"] = evaluate_model(
    three_class_trainer,
    three_class_tokenizer,
    "three_class",
    splits["three_class"]["test"],
    "Exp3 train three_class: test three_class",
)
exp3_results["without_marker_binary_test_as_three_class"] = evaluate_model(
    three_class_trainer,
    three_class_tokenizer,
    "three_class",
    binary_to_three_class(splits["without_marker_binary"]["test"], metaphor_class_label=2),
    "Exp3 train three_class: test without_marker binary as three_class",
)
exp3_results["with_marker_binary_test_as_three_class"] = evaluate_model(
    three_class_trainer,
    three_class_tokenizer,
    "three_class",
    binary_to_three_class(splits["with_marker_binary"]["test"], metaphor_class_label=1),
    "Exp3 train three_class: test with_marker binary as three_class",
)


## 14. Attention Heatmap

This visualizes token-level attention from the final transformer layer. For research reporting, treat attention as a diagnostic visualization, not a complete explanation of model reasoning.

In [ ]:
def plot_attention_heatmap(text, model_dir, layer=-1, head=0, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        output_attentions=True,
        attn_implementation="eager",
    ).to(DEVICE)
    model.eval()

    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(DEVICE)
    with torch.no_grad():
        outputs = model(**encoded)

    pred_id = int(outputs.logits.argmax(dim=-1).item())
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    attention = outputs.attentions[layer][0, head].detach().cpu().numpy()

    plt.figure(figsize=(max(8, len(tokens) * 0.45), max(6, len(tokens) * 0.35)))
    sns.heatmap(attention, xticklabels=tokens, yticklabels=tokens, cmap="viridis")
    plt.xticks(rotation=60, ha="right")
    plt.yticks(rotation=0)
    plt.title(f"Attention heatmap | predicted label: {pred_id}")
    plt.tight_layout()
    plt.show()

    return pred_id


# Run attention heatmap on 3 samples from each experiment's test set
for _ds_name, _trainer_tok in [
    ("with_marker_binary",   (with_marker_trainer,   with_marker_tokenizer)),
    ("without_marker_binary",(without_marker_trainer, without_marker_tokenizer)),
    ("three_class",          (three_class_trainer,   three_class_tokenizer)),
]:
    _model_dir = OUTPUT_DIR / _ds_name / "best_model"
    _samples = splits[_ds_name]["test"]["line"].head(3).tolist()
    print(f"\n=== Attention heatmaps: {_ds_name} ===")
    for _text in _samples:
        plot_attention_heatmap(_text, _model_dir, layer=-1, head=0)


## 15. LIME Interpretability

LIME explains one prediction by perturbing words and observing how prediction probabilities change.

In [ ]:
def make_predict_proba(model_dir, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    model.eval()

    def predict_proba(texts):
        encoded = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(**encoded).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        return probs

    return predict_proba, model.config.id2label


def explain_with_lime(text, model_dir, num_features=10):
    predict_proba, id2label = make_predict_proba(model_dir)
    class_names = [id2label[i] for i in range(len(id2label))]
    explainer = LimeTextExplainer(class_names=class_names)
    explanation = explainer.explain_instance(
        text,
        predict_proba,
        num_features=num_features,
        labels=list(range(len(class_names))),
    )
    explanation.show_in_notebook(text=True)
    return explanation


# Run LIME on 3 samples from each experiment's test set
for _ds_name in ["with_marker_binary", "without_marker_binary", "three_class"]:
    _model_dir = OUTPUT_DIR / _ds_name / "best_model"
    _samples = splits[_ds_name]["test"]["line"].head(3).tolist()
    print(f"\n=== LIME explanations: {_ds_name} ===")
    for _text in _samples:
        print(f"\nSample: {_text}")
        explain_with_lime(_text, _model_dir)


## 16. SHAP Interpretability

SHAP gives token contribution estimates. Use a small sample first because SHAP can be slow for transformer models.

In [ ]:
def explain_with_shap(texts, model_dir, max_examples=5):
    texts = list(texts)[:max_examples]
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    model.eval()

    def predict_proba(batch_texts):
        encoded = tokenizer(
            list(batch_texts),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(**encoded).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        return probs

    masker = shap.maskers.Text(tokenizer)
    explainer = shap.Explainer(predict_proba, masker)
    shap_values = explainer(texts)
    shap.plots.text(shap_values)
    return shap_values


# Run SHAP on 3 samples from each experiment's test set
# (SHAP can be slow for transformers; keep max_examples small)
for _ds_name in ["with_marker_binary", "without_marker_binary", "three_class"]:
    _model_dir = OUTPUT_DIR / _ds_name / "best_model"
    _sample_texts = splits[_ds_name]["test"]["line"].head(3).tolist()
    print(f"\n=== SHAP explanations: {_ds_name} ===")
    explain_with_shap(_sample_texts, _model_dir, max_examples=3)


## 17. Suggested Reporting Table

After the experiments finish, compare `macro_f1`, `macro_precision`, `macro_recall`, and class-wise F1 from the saved classification reports. The most important comparisons are cross-domain drops: marker-trained model on markerless test data, and markerless-trained model on marker-based test data.